# Practica final NLP - Ejercicio 3
## Modelos de sentimiento

Usamos **Bag of Words** con `CountVectorizer` y entrenamos dos modelos
clasicos: **Multinomial Naive Bayes** y **Logistic Regression**.

## Librerias

In [1]:
import pandas as pd

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

## 1. Cargar el dataset preprocesado

In [2]:
df = pd.read_csv('reviews_preprocessed.csv')
df = df.dropna(subset=['clean_text'])
print('Tamano:', df.shape)
print('Balanceo de clases:')
print(df['sentiment'].value_counts())
df.head()

Tamano: (9993, 2)
Balanceo de clases:
sentiment
0    4997
1    4996
Name: count, dtype: int64


,clean_text,sentiment
0,poorly remastered problem remastering poor sou...,0
1,great collection time life whole collection pu...,1
2,five star best series ever love everyone kink ...,1
3,one best live performance ever one best live p...,1
4,two exclusive vault track byrd columbia record...,1


## 2. Train / Test split

In [3]:
X = df['clean_text']
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print('Train:', X_train.shape, 'Test:', X_test.shape)

Train: (7994,) Test: (1999,)


## 3. Bag of Words con CountVectorizer

Parametros:
- `max_features=5000`: limitamos el vocabulario a las 5000 palabras mas
  frecuentes para que la matriz no sea demasiado grande.
- `min_df=2`: ignoramos palabras que aparecen en menos de 2 documentos.

In [4]:
vectorizer = CountVectorizer(max_features=5000, min_df=2)
X_train_bow = vectorizer.fit_transform(X_train)
X_test_bow = vectorizer.transform(X_test)
print('Forma de la matriz BoW:', X_train_bow.shape)

Forma de la matriz BoW: (7994, 5000)


## 4. Modelo 1: Multinomial Naive Bayes

In [5]:
nb = MultinomialNB()
nb.fit(X_train_bow, y_train)
pred_nb = nb.predict(X_test_bow)

print('Accuracy: ', accuracy_score(y_test, pred_nb))
print('Precision:', precision_score(y_test, pred_nb))
print('Recall:   ', recall_score(y_test, pred_nb))
print('F1:       ', f1_score(y_test, pred_nb))

Accuracy:  0.8579289644822411
Precision: 0.8427612655800575
Recall:    0.8798798798798799
F1:        0.8609206660137121


## 5. Modelo 2: Logistic Regression

In [6]:
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train_bow, y_train)
pred_lr = lr.predict(X_test_bow)

print('Accuracy: ', accuracy_score(y_test, pred_lr))
print('Precision:', precision_score(y_test, pred_lr))
print('Recall:   ', recall_score(y_test, pred_lr))
print('F1:       ', f1_score(y_test, pred_lr))

Accuracy:  0.8704352176088044
Precision: 0.8648915187376726
Recall:    0.8778778778778779
F1:        0.8713363139592648


## 6. Comparacion de los dos modelos

In [7]:
resultados = pd.DataFrame({
    'Naive Bayes': [
        accuracy_score(y_test, pred_nb),
        precision_score(y_test, pred_nb),
        recall_score(y_test, pred_nb),
        f1_score(y_test, pred_nb),
    ],
    'Logistic Regression': [
        accuracy_score(y_test, pred_lr),
        precision_score(y_test, pred_lr),
        recall_score(y_test, pred_lr),
        f1_score(y_test, pred_lr),
    ],
}, index=['Accuracy', 'Precision', 'Recall', 'F1'])
resultados

,Naive Bayes,Logistic Regression
Accuracy,0.857929,0.870435
Precision,0.842761,0.864892
Recall,0.879880,0.877878
F1,0.860921,0.871336


## 7. Eleccion del mejor modelo

Los dos modelos dan resultados muy parecidos. Nos quedamos con el que
tiene mejor F1 (en las pruebas Logistic Regression sale un poco
mejor).